un'azienda di e-commerce vuole analizzare i comment dei clienti. Dato che i commeni sono brevi ma ricchi di sfrumature (es. 'il prodotto non è male, ma la spedizione è stata lenta'), una RNN unidirezionale potrebbe perdere il contesto delle negazioni poste a fine frase.
Crea un modello che utilizzi un layer Bidirectinal avvolto attorno a una cella GRU con 64 unità
Utilizza una concatenazione come modalità di fusione.
Aggiungi un layer Dropout al 30% dopo il layer bidirezionale per evitare l'overfitting.
Stampa il numero totale di parametri e spiega perchè è superiore rispetto a una GRU unidirezionale.


In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import keras
from keras import layers

# 1. GENERAZIONE DATI SINTETICI
num_samples = 1000
seq_len = 10
embedding_dim = 8

X = np.random.rand(num_samples, seq_len, embedding_dim).astype("float32")
y = np.random.randint(0, 2, size=(num_samples, 1)).astype("float32")

# 2. MODELLO UNIDIREZIONALE (Baseline)
# Modifica: Passaggio a GRU con 64 unità
model_uni = keras.Sequential([
    layers.Input(shape=(seq_len, embedding_dim)),
    layers.GRU(64), 
    layers.Dense(1, activation='sigmoid')
], name="Unidirectional_GRU")

# 3. MODELLO BIDIREZIONALE (E-commerce Analysis)
# Modifica: Wrapper Bidirectional su GRU(64) + Dropout(0.3)
# 
model_bi = keras.Sequential([
    layers.Input(shape=(seq_len, embedding_dim)),
    layers.Bidirectional(layers.GRU(64), merge_mode='concat'),
    layers.Dropout(0.3), # Previene l'overfitting sui commenti brevi
    layers.Dense(1, activation='sigmoid')
], name="Bidirectional_GRU")

# 4. ANALISI E CONFRONTO (Utilizziamo la tua funzione preesistente)
def compare_accuracy(model1, model2, data_X, data_y, epochs=10):
    print(f"\n Training su Backend: {keras.backend.backend()}")
    model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    print(f"Addestramento {model1.name}...")
    model1.fit(data_X, data_y, epochs=epochs, batch_size=32, verbose=0)
    
    print(f"Addestramento {model2.name}...")
    model2.fit(data_X, data_y, epochs=epochs, batch_size=32, verbose=0)

# ESECUZIONE
compare_accuracy(model_uni, model_bi, X, y, epochs=5)

# 5. RIEPILOGO PARAMETRI
print("\n" + "="*40)
print("CONFRONTO COMPLESSITÀ")
print("="*40)
print(f"Parametri GRU Unidirezionale: {model_uni.count_params():,}")
print(f"Parametri GRU Bidirezionale:  {model_bi.count_params():,}")


 Training su Backend: torch
Addestramento Unidirectional_GRU...
Addestramento Bidirectional_GRU...

CONFRONTO COMPLESSITÀ
Parametri GRU Unidirezionale: 14,273
Parametri GRU Bidirezionale:  28,545
